# DPO 训练一键脚本 — llm-preference-alignment-dpo

**跑之前先做两件事：**

1. 把修好的 `scripts/` 和 `requirements.txt` 推到 GitHub（这个 notebook 是从仓库 clone 代码的）
2. 上方菜单 **代码执行程序 → 更改运行时类型 → T4 GPU** → 保存

然后 **代码执行程序 → 全部运行**，大约 25-35 分钟跑完。

最后一个 cell 会打印一段可以直接粘贴进 README `## Results` 的 Markdown。

## 0. 确认拿到 GPU

In [ ]:
!nvidia-smi

import torch
assert torch.cuda.is_available(), "没有 GPU：代码执行程序 → 更改运行时类型 → T4 GPU"
cap = torch.cuda.get_device_capability()
print(f"\nGPU: {torch.cuda.get_device_name(0)} | compute capability {cap[0]}.{cap[1]}")
print(f"is_bf16_supported() 说: {torch.cuda.is_bf16_supported()}  <- T4 上这个是骗人的（算的是软件模拟）")
print(f"脚本实际会用: {'bf16' if cap[0] >= 8 else 'fp16'}  <- 按算力等级判断，T4 (7.5) 必须走 fp16")

## 1. 装依赖（约 2 分钟）

In [ ]:
# 版本锁死到我在容器里端到端验证过的组合。
# 之前用的是版本区间，Colab 预装的旧 peft 满足区间就没被升级，
# 和 trl 1.12 对不上，于是 get_peft_model 直接炸了。
!pip install -q "transformers==5.16.1" "trl==1.12.0" "peft==0.20.0" \
    "datasets==5.0.1" "accelerate==1.14.0"

import transformers, trl, peft, datasets, accelerate
want = {"transformers": "5.16.1", "trl": "1.12.0", "peft": "0.20.0",
        "datasets": "5.0.1", "accelerate": "1.14.0"}
for m in (transformers, trl, peft, datasets, accelerate):
    mark = "OK " if m.__version__ == want[m.__name__] else "!! "
    print(f"{mark}{m.__name__:14s} {m.__version__}  (期望 {want[m.__name__]})")

print("\n如果上面有 !! ，先点『代码执行程序 → 重新启动会话』再从这个 cell 往下跑一遍。")

## 2. 拉代码

In [ ]:
REPO   = "https://github.com/ztitus720/llm-preference-alignment-dpo.git"
BRANCH = "main"

import os, shutil
os.chdir("/content")          # 先离开要删的目录，否则删掉 cwd 会让 shell 的路径悬空
shutil.rmtree("/content/dpo", ignore_errors=True)

!git clone -q -b $BRANCH $REPO /content/dpo
%cd /content/dpo
!git log -1 --format="%h %ad %s" --date=short
!ls scripts/

assert os.path.isfile("scripts/common.py"), (
    "仓库里没有 scripts/common.py —— 修好的脚本还没 push 上去。\n"
    "在本地 clone 里解压 dpo_fixed.zip 覆盖，然后 git add -A && git commit && git push，"
    "再回来重跑这个 cell。"
)
print("\nOK: 拿到的是修好的版本")

## 3. 构造偏好数据

默认从 `trl-lib/ultrafeedback_binarized` 采 300 对。

如果这个数据集报错，把下面换成：
`--dataset HuggingFaceH4/ultrafeedback_binarized --split train_prefs`

In [ ]:
!python scripts/build_preference_data.py --source hf --n 300

!head -c 600 data/preference_pairs.jsonl; print()
!wc -l data/preference_pairs.jsonl

## 4. 训练（T4 上约 20-30 分钟）

显存不够就把 `--model` 换成 `Qwen/Qwen2.5-0.5B-Instruct`，或者 `--max_length 384`。

其中 15% 的数据会留作 held-out，用来算下一步那个数字。

In [ ]:
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"   # OOM 就改成 Qwen/Qwen2.5-0.5B-Instruct

!python scripts/train_dpo.py \
    --model $MODEL \
    --epochs 1 \
    --batch_size 2 \
    --grad_accum 8 \
    --max_length 512 \
    --beta 0.1 \
    --output_dir ./dpo-out

## 5. 评测 — 这一步产出简历上那个数字

对每一对 held-out 数据，分别算 chosen / rejected 在 base 和 DPO 模型下的对数似然，得到：

- **preference accuracy**：模型把 chosen 排在 rejected 前面的比例（base → tuned 的变化才是结果），同时给出长度归一化版本，避免"短答案天然占便宜"
- **DPO implicit-reward accuracy / margin**：DPO 真正优化的那个量

不需要 OpenAI key，别人 clone 下来能复现同一个数字。

In [ ]:
!python scripts/evaluate.py \
    --base $MODEL \
    --tuned ./dpo-out \
    --pairs ./dpo-out/heldout_pairs.jsonl \
    --beta 0.1

## 6. 生成可以直接粘贴进 README 的 Results 段落

In [ ]:
import json, datetime, torch

m = json.load(open("eval_metrics.json"))
t = json.load(open("dpo-out/train_metrics.json"))
n_pairs = sum(1 for _ in open("data/preference_pairs.jsonl"))

block = f"""## Results

Run on {datetime.date.today().isoformat()}, single {torch.cuda.get_device_name(0)} (Colab free tier).

| | value |
|---|---|
| Base model | `{MODEL}` |
| Preference data | `trl-lib/ultrafeedback_binarized`, {n_pairs} pairs ({m['n_pairs']} held out) |
| Training | LoRA r=16 on q/k/v/o, 1 epoch, lr 5e-6, beta 0.1, {t.get('train_runtime', 0)/60:.1f} min |
| Final training loss | {t.get('train_loss', float('nan')):.4f} |

**Held-out preference accuracy** (does the model rank the preferred answer above the rejected one?)

| | base | DPO-tuned |
|---|---|---|
| sum log-prob | {m['base_preference_accuracy']:.1%} | {m['tuned_preference_accuracy']:.1%} |
| length-normalised | {m['base_preference_accuracy_len_norm']:.1%} | {m['tuned_preference_accuracy_len_norm']:.1%} |

DPO implicit reward `r(y|x) = beta * (log p_tuned - log p_base)` on the same held-out pairs:
**accuracy {m['dpo_implicit_reward_accuracy']:.1%}**, mean margin {m['dpo_implicit_reward_margin']:.4f}
(TRL reported `eval_rewards/accuracies` = {t.get('eval_rewards/accuracies', float('nan')):.3f}, computed independently — the two agree.)

Mean response length in the held-out set: {m['mean_len_chosen']:.0f} tokens chosen vs {m['mean_len_rejected']:.0f} rejected,
so the length-normalised row above is the one to trust if the two differ much.

Side-by-side generations on five fixed prompts: `eval_results.jsonl`.
Raw metrics: `eval_metrics.json`, `dpo-out/train_metrics.json`.
"""

print(block)
open("RESULTS_SNIPPET.md", "w").write(block)

## 7. 把结果下载下来（adapter + 两个 metrics json）

In [ ]:
!zip -qr dpo_results.zip dpo-out eval_metrics.json eval_results.jsonl RESULTS_SNIPPET.md

from google.colab import files
files.download("dpo_results.zip")

## 8.（可选，约 10 分钟）自采样 + rubric 造数据

上面用的是现成的开源偏好数据集。这一步让模型自己在两个温度下各采一次，再用 rubric 打标 —— 对应 Qwen JD 里"高质量数据合成"那条。

跑完可以把 `--data` 指向它再训一版，对比两种数据来源的效果差异，这个对比本身就是面试可讲的内容。

In [ ]:
# !python scripts/build_preference_data.py --source synthetic --n 100 \
#     --model Qwen/Qwen2.5-0.5B-Instruct --out data/synthetic_pairs.jsonl
# !python scripts/train_dpo.py --model $MODEL --data data/synthetic_pairs.jsonl \
#     --output_dir ./dpo-out-synthetic --epochs 1